# 07. 表示順ランダム化と再検証

## 経緯

`06_pipeline_selftest.ipynb`で符号チェックは通過したが、実際の自己回答テストで
Q1〜Q8の8問すべてで同じ側（前者）を選ぶという結果になった。内容で答えていた可能性も
残るが、選択肢の並び順を固定していたこと自体が心理尺度の基本に反しており、原因が
反応バイアスか実際の傾向かをこの並びのままでは判別できない。

対応として、質問データを「表示位置（前者/後者）」ではなく「内容（選択肢A/B）」で
管理するように変更し、表示順をシード付きでランダム化する。採点は常に内容ベースで
行うため、表示順に関わらず正しく採点される。

前回（表示順固定）の回答は履歴として保持し、ランダム化後に日を空けて再回答した
結果と比較する。同じプロファイルが出れば実際の傾向、変われば並び順に引きずられていた
ことになる。


In [1]:
import hashlib
import random
from datetime import date

import pandas as pd

pd.set_option("display.max_colwidth", 80)
pd.set_option("display.width", 200)


## 質問データ：内容（選択肢A/B）で管理する

`positive`は"a"か"b"で、その選択肢がPC軸の正の方向に対応することを示す。表示順とは
無関係な、内容そのものの識別子。


In [2]:
QUESTIONS = [
    {"id": "Q1", "axis": "PC1", "option_a": "テストのように答えが一つに決まっている問題の方が面白い", "option_b": "感想文のように人それぞれ答えが違う問題の方が面白い", "positive": "b"},
    {"id": "Q2", "axis": "PC1", "option_a": "教わったとおりにやる方が好き", "option_b": "自分なりのやり方を試す方が好き", "positive": "b"},
    {"id": "Q3", "axis": "PC1", "option_a": "レシピどおりに正確に作る方が好き", "option_b": "レシピを見ながらも自分なりにアレンジする方が好き", "positive": "b"},
    {"id": "Q4", "axis": "PC2", "option_a": "こわれたものを自分で直してみたい", "option_b": "友達のけんかの仲直りを手伝いたい", "positive": "a"},
    {"id": "Q5", "axis": "PC2", "option_a": "新しい道具や機械のしくみを調べる方が得意な気がする", "option_b": "友達や家族の悩みを聞いてあげる方が得意な気がする", "positive": "a"},
    {"id": "Q6", "axis": "PC2", "option_a": "作り方の動画を見て何かを作る方が楽しそう", "option_b": "人の話を聞いてアドバイスする方が楽しそう", "positive": "a"},
    {"id": "Q7", "axis": "PC3", "option_a": "最初に完成までの手順を決めてから進める方が自分に近い", "option_b": "進めながら決めていく方が自分に近い", "positive": "a"},
    {"id": "Q8", "axis": "PC3", "option_a": "ゲームはルールどおりに進める方が好き", "option_b": "自分たちでルールを変えて遊ぶ方が好き", "positive": "a"},
    {"id": "Q9", "axis": "PC4", "option_a": "表現の美しさ・かっこよさにこだわる授業の方が好き", "option_b": "しくみや理由を突き止める授業の方が好き", "positive": "a"},
    {"id": "Q10", "axis": "PC4", "option_a": "作品を作って見せる方にする", "option_b": "調べて分かったことをまとめる方にする", "positive": "a"},
]
AXIS_N = {"PC1": 3, "PC2": 3, "PC3": 2, "PC4": 2}


## 表示順のシャッフル（シード付き、再現可能）

`seed`が同じなら、同じ並びが再現される。採点は`option_a`/`option_b`の内容で行うため、
どちらが先に表示されたかには依存しない。


In [3]:
def display_order(question_id: str, seed: str) -> tuple[str, str]:
    """("a","b") か ("b","a") を返す。表示上、先頭に出す方が最初の要素。"""
    h = hashlib.sha256(f"{seed}:{question_id}".encode()).hexdigest()
    rnd = random.Random(h)
    return ("a", "b") if rnd.random() < 0.5 else ("b", "a")


def render_question(q: dict, seed: str) -> dict:
    first, second = display_order(q["id"], seed)
    labels = {"a": q["option_a"], "b": q["option_b"]}
    return {
        "id": q["id"],
        "text": q.get("text", q["axis"]),
        "choice_1": labels[first],  # 画面上1番目
        "choice_2": labels[second],  # 画面上2番目
    }


SEED = "session-2026-09-09"
for q in QUESTIONS:
    r = render_question(q, SEED)
    print(f"{r['id']}: 1番目=[{r['choice_1'][:20]}...] 2番目=[{r['choice_2'][:20]}...]")


Q1: 1番目=[テストのように答えが一つに決まっている問...] 2番目=[感想文のように人それぞれ答えが違う問題の...]
Q2: 1番目=[教わったとおりにやる方が好き...] 2番目=[自分なりのやり方を試す方が好き...]
Q3: 1番目=[レシピどおりに正確に作る方が好き...] 2番目=[レシピを見ながらも自分なりにアレンジする...]
Q4: 1番目=[友達のけんかの仲直りを手伝いたい...] 2番目=[こわれたものを自分で直してみたい...]
Q5: 1番目=[新しい道具や機械のしくみを調べる方が得意...] 2番目=[友達や家族の悩みを聞いてあげる方が得意な...]
Q6: 1番目=[作り方の動画を見て何かを作る方が楽しそう...] 2番目=[人の話を聞いてアドバイスする方が楽しそう...]
Q7: 1番目=[進めながら決めていく方が自分に近い...] 2番目=[最初に完成までの手順を決めてから進める方...]
Q8: 1番目=[自分たちでルールを変えて遊ぶ方が好き...] 2番目=[ゲームはルールどおりに進める方が好き...]
Q9: 1番目=[表現の美しさ・かっこよさにこだわる授業の...] 2番目=[しくみや理由を突き止める授業の方が好き...]
Q10: 1番目=[作品を作って見せる方にする...] 2番目=[調べて分かったことをまとめる方にする...]


## 採点：内容ベース（表示順に依存しない）

回答は「選んだ選択肢の文言そのもの」で受け取り、`option_a`/`option_b`と照合して
a/bを判定する。表示順が入れ替わっていても正しく採点される。


In [4]:
def match_choice(q: dict, chosen_text: str) -> str:
    """回答文言をa/bに正規化する。"""
    if chosen_text.strip() == q["option_a"].strip():
        return "a"
    if chosen_text.strip() == q["option_b"].strip():
        return "b"
    raise ValueError(f"{q['id']}: 選択肢と一致しない回答: {chosen_text!r}")


def score_responses(responses: dict) -> dict:
    """responses: {question_id: 選んだ選択肢の文言}"""
    raw = {axis: 0 for axis in AXIS_N}
    for q in QUESTIONS:
        ab = match_choice(q, responses[q["id"]])
        sign = 1 if ab == q["positive"] else -1
        raw[q["axis"]] += sign
    return raw


## 回答履歴を保存する

前回（表示順固定、`06_pipeline_selftest.ipynb`で実施）の回答を、内容ベースの記録として
残す。今後、表示順をランダム化した状態で日を空けて再回答した結果をここに追記して比較する。


In [5]:
import os

RESPONSES_PATH = "../data/processed/self_test_responses.csv"

# 前回（06で実施、表示順固定＝前者を常に選択肢の1番目として提示していた）の回答を
# 内容（選んだ選択肢の文言）として記録する
session1_choices = {
    "Q1": "テストのように答えが一つに決まっている問題の方が面白い",
    "Q2": "教わったとおりにやる方が好き",
    "Q3": "レシピどおりに正確に作る方が好き",
    "Q4": "こわれたものを自分で直してみたい",
    "Q5": "新しい道具や機械のしくみを調べる方が得意な気がする",
    "Q6": "作り方の動画を見て何かを作る方が楽しそう",
    "Q7": "最初に完成までの手順を決めてから進める方が自分に近い",
    "Q8": "ゲームはルールどおりに進める方が好き",
    "Q9": "しくみや理由を突き止める授業の方が好き",
    "Q10": "作品を作って見せる方にする",
}

rows = []
if os.path.exists(RESPONSES_PATH):
    history = pd.read_csv(RESPONSES_PATH)
    rows = history.to_dict("records")

session1_id = "2026-09-09_session1_表示順固定"
if not any(r["session_id"] == session1_id for r in rows):
    for qid, text in session1_choices.items():
        rows.append({"session_id": session1_id, "question_id": qid, "chosen_text": text,
                     "note": "初回・表示順固定（前者を常に1番目に提示）。ランダム化前の記録"})

history_df = pd.DataFrame(rows)
history_df.to_csv(RESPONSES_PATH, index=False, encoding="utf-8-sig")
print("保存件数:", len(history_df))
history_df


保存件数: 10


,session_id,question_id,chosen_text,note
0,2026-09-09_session1_表示順固定,Q1,テストのように答えが一つに決まっている問題の方が面白い,初回・表示順固定（前者を常に1番目に提示）。ランダム化前の記録
1,2026-09-09_session1_表示順固定,Q2,教わったとおりにやる方が好き,初回・表示順固定（前者を常に1番目に提示）。ランダム化前の記録
2,2026-09-09_session1_表示順固定,Q3,レシピどおりに正確に作る方が好き,初回・表示順固定（前者を常に1番目に提示）。ランダム化前の記録
3,2026-09-09_session1_表示順固定,Q4,こわれたものを自分で直してみたい,初回・表示順固定（前者を常に1番目に提示）。ランダム化前の記録
4,2026-09-09_session1_表示順固定,Q5,新しい道具や機械のしくみを調べる方が得意な気がする,初回・表示順固定（前者を常に1番目に提示）。ランダム化前の記録
5,2026-09-09_session1_表示順固定,Q6,作り方の動画を見て何かを作る方が楽しそう,初回・表示順固定（前者を常に1番目に提示）。ランダム化前の記録
6,2026-09-09_session1_表示順固定,Q7,最初に完成までの手順を決めてから進める方が自分に近い,初回・表示順固定（前者を常に1番目に提示）。ランダム化前の記録
7,2026-09-09_session1_表示順固定,Q8,ゲームはルールどおりに進める方が好き,初回・表示順固定（前者を常に1番目に提示）。ランダム化前の記録
8,2026-09-09_session1_表示順固定,Q9,しくみや理由を突き止める授業の方が好き,初回・表示順固定（前者を常に1番目に提示）。ランダム化前の記録
9,2026-09-09_session1_表示順固定,Q10,作品を作って見せる方にする,初回・表示順固定（前者を常に1番目に提示）。ランダム化前の記録


## 初回（表示順固定）の結果を、内容ベースの採点ロジックで再計算する

採点ロジックの変更が結果を変えていないことを確認する（表示順の情報を使わず、内容だけで
同じ結果が出るはず）。


In [6]:
raw_session1 = score_responses(session1_choices)
print("軸ごとの生スコア:", raw_session1)
assert raw_session1 == {"PC1": -3, "PC2": 3, "PC3": 2, "PC4": 0}, "06での手計算結果と不一致"
print("06での結果と一致：採点ロジックの変更は結果に影響していない")


軸ごとの生スコア: {'PC1': -3, 'PC2': 3, 'PC3': 2, 'PC4': 0}
06での結果と一致：採点ロジックの変更は結果に影響していない


## 次のステップ（このノートブックでは実施しない）

日を空けて、`render_question`でシャッフルした状態の10問に再回答し、`session_id`を
変えて`self_test_responses.csv`に追記する。2回の結果を比較し、初回のPC1〜PC3の
一方向への偏りが表示順によるものだったか、実際の傾向だったかを判定する。
